In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

def preprocess(train_file, test_file):
    train = np.load(train_file, allow_pickle=True)
    test = np.load(test_file, allow_pickle=True)

    X_train, y_train = train["X"], train["y"]
    print(X_train.shape, y_train.shape)
    X_test, y_test = test["X"], test["y"]
    print(X_test.shape, y_test.shape)

    print(X_train[2].shape)
    print(len(np.unique(y_train)))

    H = W = 32

    transform = transforms.Compose([
        transforms.Resize(size=(H, W), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor()
    ])

    train_data = []
    for img in X_train:
        img = Image.fromarray(img)
        img = transform(img)
        train_data.append(img)

    test_data = []
    for img in X_train:
        img = Image.fromarray(img)
        img = transform(img)
        test_data.append(img)

    X_train = torch.stack(train_data)
    X_test  = torch.stack(test_data)

    return (X_train, y_train), (X_test, y_test)


(X_train, y_train), (X_test, y_test) = preprocess("train2.npz", "test2.npz")

(60000,) (60000,)
(10000,) (10000,)
(30, 44)
10
torch.Size([1, 32, 32])


In [12]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.act = nn.ReLU()

        # IN -> (1, 32, 32)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1) # -> (16, 32, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2) # -> (16, 16, 16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, padding=1) # -> (16, 16, 16)
        self.pool2 = nn.MaxPool2d(kernel_size=2) # -> (16, 8, 8)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=8, kernel_size=3, padding=1) # -> (8, 8, 8)
        self.pool3 = nn.MaxPool2d(kernel_size=2) # -> (8, 4, 4)

        self.flat = nn.Flatten()

        self.fc1 = nn.Linear(in_features=8 * 4 * 4, out_features=128)
        #self.drop1 = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=128, out_features=64)
        #self.drop1 = nn.Dropout(p=0.3)
        self.fc3 = nn.Linear(in_features=64, out_features=10)

    def forward(self, x):
        x = self.pool1(self.act(self.conv1(x)))
        x = self.pool2(self.act(self.conv2(x)))
        x = self.pool3(self.act(self.conv3(x)))

        x = self.flat(x)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)

        return x


model = Model()

# Foarte buna pentru cand nu stii sa calculezi astea
# from torchsummary import summary
# summary(model, (1, 32, 32), device="cpu")

In [ ]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15

best = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )
    if test_acc > best:
        best = test_acc
        torch.save(model.state_dict(), "model2.pth")

/tmp/ipykernel_12841/424669210.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train, dtype=torch.float32),
/tmp/ipykernel_12841/424669210.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_test, dtype=torch.float32),


Epoch 1/15 | Train Loss: 0.4080 | Train Acc: 86.39% | Test Loss: 0.1285 | Test Acc: 96.06%
Epoch 2/15 | Train Loss: 0.1194 | Train Acc: 96.28% | Test Loss: 0.0788 | Test Acc: 97.48%
Epoch 3/15 | Train Loss: 0.0887 | Train Acc: 97.12% | Test Loss: 0.0651 | Test Acc: 97.87%
Epoch 4/15 | Train Loss: 0.0716 | Train Acc: 97.73% | Test Loss: 0.0668 | Test Acc: 97.93%
